# Prolog Program Explainer using LangChain

This notebook creates a tool that provides natural language explanations of Prolog programs using LangChain.

In [1]:
# Import necessary libraries
from langchain.prompts import PromptTemplate
from pydantic import BaseModel, Field
from typing import List
from langchain.output_parsers import PydanticOutputParser
from langchain.schema import StrOutputParser
from langchain_ollama import OllamaLLM
import shutil
import re
import json

In [2]:
ollama_exists = shutil.which('ollama') is not None

if not ollama_exists:
    # Install Ollama (if not already installed)
    !curl -fsSL https://ollama.com/install.sh | sh

    # Start the Ollama server
    !ollama serve

In [ ]:
# Define the output structure with a Pydantic model
class PrologExplanation(BaseModel):
    summary: str = Field(description="A brief summary of what the Prolog program does")
    primitive: List[dict] = Field(description="List of dependent primitives and their descriptions")
    explanation: str = Field(description="How the program logic works")
    
def explain_prolog(llm, prompt_template, prolog_code, variables=['prolog_code']):
    """Generate a natural language explanation of a Prolog program.
    
    Args:
        llm: The language model to use
        prompt: The prompt template to use
        prolog_code (str): The Prolog code to explain
        
    Returns:
        PrologExplanation: Structured explanation of the Prolog program
    """
    # Initialize the output parser
    parser = PydanticOutputParser(pydantic_object=PrologExplanation)
    
    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=variables,
        partial_variables={"JSON_format": parser.get_format_instructions()}
    )
    
    # Get raw string response from the LLM
    chain = prompt | llm | StrOutputParser()
    raw_result = chain.invoke({"prolog_code": prolog_code})
    
    print(f"===== Generated Raw Text =====\n{raw_result}")
    
    # Parse the raw string into a structured PrologExplanation object
    try:
        # Try to parse the output directly
        parsed_result = parser.parse(raw_result)
        
        # print(f"\n===== Parsed Prolog Explanation =====\n{parsed_result}")
        print(f"\n===== Parsed Text =====")

        print(f"Summary: {parsed_result.summary}\n")
        print("Primitives:")
        for pred in parsed_result.primitive:
            print(f"- {pred.get('name')}: {pred.get('description')}")
        print(f"\nExplanation: {parsed_result.explanation}\n")
        return parsed_result
    except Exception as e:
        # If direct parsing fails, attempt to extract structured data from text
        print(f"Parsing error: {e}")
        print("Returning raw output instead. Check format of LLM response.")
        return raw_result

In [4]:
# Llama 3.2 3B
llm = OllamaLLM(
    model = 'llama3.2'
)

In [5]:
explanation_template = """
You are an expert Prolog interpreter who explains what the Prolog code does in English to non-programmers.

I. Analyze the following Prolog program which is divided into three sections:

1. Target rules: The main predicates that define the goal of the program
2. Primitives: Supporting predicates that help implement the target rules
3. Biases: Input and output signature directions for the primitives

{prolog_code}

II. Your response MUST follow the below JSON format EXACTLY with no additional text:

{JSON_format}

YOUR EXPLANATION MUST:

In "summary": Provide a 1-2 sentence overview of what the entire Prolog program accomplishes.
In "primitive": List each primitive from Target rules and explain what they do as dictionaries with "name" and "description" keys.
In "explanation": Explain what the Target rules do to a person without any programming or technical experience.

IMPORTANT FORMATTING RULES:

Return ONLY valid parseable JSON with the exact field names shown above.
Ensure all strings are properly escaped for valid JSON.
DO NOT return JSON schema information or "properties" fields.
Do not include markdown formatting symbols, code blocks, or explanatory text outside the JSON structure.
"""

## Example Usage

In [9]:
# Example Prolog code - Family relationships
family_prolog = """
%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%
parent(X, Y) :- father(X, Y).
parent(X, Y) :- mother(X, Y).

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%
# direction(grandparent,(in,out)).
direction(parent,(in,out)). 
direction(father,(in,out)). 
direction(mother,(in,out)). 
"""

explanation = explain_prolog(llm, explanation_template, family_prolog)

===== Generated Raw Text =====
{"summary": "This Prolog program determines the grandparent of a person based on their parent and grandparent relationships.", "primitive": [{"name": "grandparent", "description": "Determines if X is the grandparent of Z."},{"name": "parent", "description": "Checks if one person is the father or mother of another."},{"name": "father", "description": "Checks for a paternal relationship between two people."},{"name": "mother", "description": "Checks for a maternal relationship between two people."}], "explanation": "The program starts by checking if X has a parent Y and then checks if Y has a child Z. If both conditions are true, then X is the grandparent of Z."}

===== Parsed Text =====
Summary: This Prolog program determines the grandparent of a person based on their parent and grandparent relationships.

Primitives:
- grandparent: Determines if X is the grandparent of Z.
- parent: Checks if one person is the father or mother of another.
- father: Checks 

## Custom Prolog Code Explanation

Without code comments

In [14]:
# Example Prolog code - List processing
list_prolog = """

%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%

inv_ho_0(A,B):- same_circuit(A,B),linear_path(B,A).
inv_ho_1(A,B):- same_circuit(A,B),not(linear_path(B,A)).

partition(A,B,C):- all(inv_ho_0,A,B),all(inv_ho_1,A,C).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%
all(P, X, L):- 
    findall(H, call(P, X, H), L).

has_branches(X) :- is_connected(X, Y), is_connected(X, Z), Y \\== Z.

linear_path(X, X).
linear_path(X, Y) :- is_connected(X, Z), not(has_branches(X)), linear_path(Z, Y).
not_linear_path(X, Y) :- not(linear_path(X, Y)).

same_circuit(X, Y) :- gate(X), gate(Y), N is X // 100, M is Y // 100, N == M.

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%
direction(partition,(in,out,out)). 
direction(linear_path,(in,out)).
direction(not_linear_path,(in,out)).
direction(same_circuit,(in,out)).
direction(all,((in,out),in,out)).

"""

explanation = explain_prolog(llm, explanation_template, list_prolog)

===== Generated Raw Text =====
{"summary": "This Prolog program analyzes the connectivity of a digital circuit and identifies linear paths and circuits with the same connection points.", "primitive": [{"name": "linear_path", "description": "A predicate that checks if there is a direct path between two nodes in the graph"}, {"name": "not_linear_path", "description": "A predicate that checks if there is no direct path between two nodes in the graph"}, {"name": "same_circuit", "description": "A predicate that checks if two circuits have the same connection points"}, {"name": "has_branches", "description": "A predicate that checks if a node has more than one branch"}, {"name": "is_connected", "description": "A predicate that checks if two nodes are connected by an edge"}, {"name": "all", "description": "A predicate that applies the same operation to all elements of a list"}], "explanation": "The program starts by defining several predicates to analyze the connectivity of the circuit. It th

With code comments

In [8]:
# Example Prolog code - List processing
list_prolog = """

%%%%%%%%%%%%%%%%%%%% Target rules %%%%%%%%%%%%%%%%%%%%

% Gates affected by a test and those not affected
inv_ho_0(A,B):- same_circuit(A,B),linear_path(B,A).
inv_ho_1(A,B):- same_circuit(A,B),not(linear_path(B,A)).

% Given Gate A find gates on the same linear path as A and gates on other paths
partition(A,B,C):- all(inv_ho_0,A,B),all(inv_ho_1,A,C).

%%%%%%%%%%%%%%%%%%%% Primitives %%%%%%%%%%%%%%%%%%%%
% Higher order primitive
all(P, X, L):- 
    findall(H, call(P, X, H), L).

% Are multiple gates connected to gate X?
has_branches(X) :- is_connected(X, Y), is_connected(X, Z), Y \\== Z.

% Is gate X to gate Y a path without branches?
linear_path(X, X).
linear_path(X, Y) :- is_connected(X, Z), not(has_branches(X)), linear_path(Z, Y).
not_linear_path(X, Y) :- not(linear_path(X, Y)).

% Does gate X share a linear_path with gate Y (Y precedes X if X \\== Y)?
same_circuit(X, Y) :- gate(X), gate(Y), N is X // 100, M is Y // 100, N == M.

%%%%%%%%%%%%%%%%%%%% Biases %%%%%%%%%%%%%%%%%%%%
direction(partition,(in,out,out)). 
direction(linear_path,(in,out)).
direction(not_linear_path,(in,out)).
direction(same_circuit,(in,out)).
direction(all,((in,out),in,out)).

"""

explanation = explain_prolog(llm, explanation_template, list_prolog)

===== Generated Raw Text =====
{"summary": "This Prolog program analyzes a digital circuit diagram and partitions it into gates on the same linear path as a given gate.", 
"primitive":
[
    {"name": "all", "description": "Finds all results from calling another predicate with a list of arguments"},
    {"name": "has_branches", "description": "Checks if multiple gates are connected to a given gate"},
    {"name": "linear_path", "description": "Determines if two gates are on the same linear path (no branches)"},
    {"name": "not_linear_path", "description": "Determine if two gates are not on the same linear path"},
    {"name": "same_circuit", "description": "Checks if two gates share a linear path"}
],
"explanation": "This Prolog program helps analyze and understand digital circuit diagrams. It takes a gate as input and finds other gates that are either affected by it (on the same linear path) or not affected by it (not on the same linear path). The program can then partition these gat